In [3]:
7 % 7

0

In [ ]:
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import numpy as np

# Load XML file
tree = ET.parse('flowmon-results.xml')  # change path if needed
root = tree.getroot()

# Store throughput data
throughput_data = []

# Convert ns to seconds
ns_to_s = 1e-9

# Parse FlowStats
for flow in root.find('FlowStats'):
    flow_id = int(flow.attrib['flowId'])
    tx_bytes = int(flow.attrib['txBytes'])
    rx_bytes = int(flow.attrib['rxBytes'])

    # Convert times to seconds
    time_first_tx = float(flow.attrib['timeFirstTxPacket'].replace('+', '').replace('ns', '')) * ns_to_s
    time_last_rx = float(flow.attrib['timeLastRxPacket'].replace('+', '').replace('ns', '')) * ns_to_s

    duration = time_last_rx - time_first_tx
    if duration <= 0:
        continue  # avoid divide-by-zero

    # Throughput in Mbps
    throughput_mbps = (rx_bytes * 8) / duration / 1e6

    throughput_data.append({
        'flowId': flow_id,
        'duration': duration,
        'tx_bytes': tx_bytes,
        'rx_bytes': rx_bytes,
        'throughput_mbps': throughput_mbps,
        'start_time': time_first_tx,
        'end_time': time_last_rx,
    })

# Sort by start_time
throughput_data.sort(key=lambda x: x['start_time'])

# Extract values for plotting/analysis
times = [f['start_time'] for f in throughput_data]
throughputs = [f['throughput_mbps'] for f in throughput_data]

# Compute statistics
avg_throughput = np.mean(throughputs)
std_throughput = np.std(throughputs)
min_throughput = np.min(throughputs)
max_throughput = np.max(throughputs)

print("Throughput Statistics (Mbps):")
print(f"- Average: {avg_throughput:.3f}")
print(f"- Std Dev: {std_throughput:.3f}")
print(f"- Min: {min_throughput:.3f}")
print(f"- Max: {max_throughput:.3f}")

# Plot throughput over time
plt.figure(figsize=(10, 6))
plt.plot(times, throughputs, marker='o', linestyle='--', color='blue')
plt.title("Per-Flow Throughput Over Time")
plt.xlabel("Start Time (s)")
plt.ylabel("Throughput (Mbps)")
plt.grid(True)
plt.tight_layout()
plt.show()
